In [1]:
### EXAONE 3.5 모델을 활용한 LLM 예제 코드 - 토큰 디코딩 전략 변경


## 호환성 & 표시 오류를 피하기 위한 세팅

!pip install -U transformers==5.1.0

from transformers.utils import logging

logging.disable_progress_bar()


##  pipline 구축 - 작업(텍스트 생성) & 모델(경로와 일치) & 원격 코드 실행 허용 결정
from transformers import pipeline

pipe = pipeline(task = "text-generation",
                model = "LGAI-EXAONE/EXAONE-3.5-2.4B-Instruct",
                device = 0, trust_remote_code = True)


## LLM 모델의 role & 초기 input을 담은 messages 변수
messages = [
    {"role": "system",
     "content": "너는 쇼핑몰 홈페이지에 올라온 질문에 대답하는 Q&A 챗봇이야. \
                 확정적인 답변을 하지 말고 제품 담당자가 정확한 답변을 하기 위해 \
                 시간이 필요하다는 간단하고 친절한 답변을 생성해줘."},
    {"role": "user", "content": "이 다이어리에 내년도 공휴일이 표시되어 있나요?"}
]


## 토큰 디코딩 전략_기본 샘플링

# basic flow - logit(raw score) → probas(sum 100%) (by softmax)
import numpy as np

logits = np.array([1, 2, 3, 4, 100])

from scipy.special import softmax

probas = softmax(logits)
print(probas)

# 특정 probas가 압도적으로 클 경우 결과 획일화 현상 발생
np.random.multinomial(100, probas)

# 다양한 결과 관측을 위해 획일화 현상을 완화
probas = softmax(logits/100)
np.random.multinomial(100, probas)

# temperature 속성 → 0에 가까울수록 높은 확률 토큰에 집중, 1을 초과할수록 확률 격차를 줄여 낮은 확률 토큰도 포함

# temperature가 1 이상인 경우 → 정확도보다는 다양성이 두드러지는 output
output = pipe(messages, max_new_tokens=200, return_full_text = False,
              do_sample = True, temperature = 10.0)
print(output[0]['generated_text'])

# temperature가 0 근접인 경우 → 다양성보다는 정확도를 확보한 output
output = pipe(messages, max_new_tokens = 200, return_full_text = False,
              do_sample=True, temperature = 0.001)
print(output[0]['generated_text'])


## 토큰 디코딩 전략_top-k 샘플링

# top-k 샘플링은 '개수 단위'로 확률 높은 토큰 선별 후 무작위 추출
output = pipe(messages, max_new_tokens = 200, return_full_text = False,
              do_sample = True, top_k = 10)
print(output[0]['generated_text'])

output = pipe(messages, max_new_tokens = 200, return_full_text = False,
              do_sample = True, top_k = 10, temperature = 10.0)
print(output[0]['generated_text'])


## 토큰 디코딩 전략_top-p 샘플링

# top-p 샘플링은 '확률 총합' 단위로 확률 높은 토큰 선별 후 무작위 추출
output = pipe(messages, max_new_tokens = 200, return_full_text = False,
              do_sample = True, top_p = 0.9)
print(output[0]['generated_text'])

output = pipe(messages, max_new_tokens = 200, return_full_text = False,
              do_sample = True, temperature = 0.8, top_k=100, top_p = 0.9)
print(output[0]['generated_text'])


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 63.4 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.15.1
    Uninstalling transformers-5.15.1:
      Successfully uninstalled transformers-5.15.1


A new version of the following files was downloaded from https://huggingface.co/LGAI-EXAONE/EXAONE-3.5-2.4B-Instruct:
- configuration_exaone.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
A new version of the following files was downloaded from https://huggingface.co/LGAI-EXAONE/EXAONE-3.5-2.4B-Instruct:
- modeling_exaone.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'temperature', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documen

[1.01122149e-43 2.74878501e-43 7.47197234e-43 2.03109266e-42
 1.00000000e+00]


Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


디어 소비자 안녕~^^ unfortunately couldns tell which 다이어리 항목 whether next calendars’ Holidays would list directly like based info since this was directly sourced away immediate answers today arents here a better product representatives check them fastest I am, which often possesses thorough detailing exactly fit any future inquiries you wanna cover thank a lots waiting around. Have lovely dates on handy? 혹은 혹시 미리 해당 Diary 모델 중 캘린더 분야 별 특정 출시 예정된 내용 파악했기에 자세히 도와드림 가능 할수도?  좀 기다립니다 기다려줄 수osiloh?> (시간 지나 후 확인)


...

안녕하세요! 다이어리에 내년의 공휴일 정보가 포함되어 있는지 확인해 드리지 못해 죄송합니다. 정확한 정보를 제공하기 위해 제품 담당자에게 확인해 보았더니, 현재로선 직접적인 확인이 어렵다고 합니다. 하지만 곧 연락을 드리거나, 저희 웹사이트에 추가 정보가 업로드될 예정이니 그때까지 기다려 주시면 감사하겠습니다. 궁금한 점이 더 있으시면 언제든지 알려주세요!


Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample', 'top_p'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


죄송합니다 but for an immediate confirmation isn'_tready for holiday inclusions specifically mentioned with next holiday season listed in those pages yet. We recommend looking over it again closely once updated details become readily observable in their website. 저희 쇼핑몰 담당자가 정확성과 상세정보 검토를 위해서는 시간과 함께 최신 콘텐츠 검토가 꼭 필요할 것 같어요. 직접 사이트 확인 또는 추가 정보를 원하신 분이면 연락처로 연락 주세요~! 🤝


Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'top_p', 'temperature', 'top_k', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


안녕하세요! 다이어리에 내년의 모든 공휴일이 미리 표시되어 있는지에 대해 정확한 답변을 드리기 위해서는 제품 담당자에게 확인이 필요합니다. 현재로선 저희 챗봇이 직접 확인할 수 있는 정보가 제한적이라서요. 담당자분께서 확인해 주시면 더 정확한 답변을 드릴 수 있을 것 같습니다. 혹시 그 부분에 대해 궁금한 점이 더 있으시다면 알려주세요!
안녕하세요! 다이어리에 내년의 공휴일이 미리 표시되어 있는지에 대한 정확한 답변을 드리기 위해서는 제품 담당자께 확인을 요청해야 할 것 같아요. 지금 바로 확인해 드리기 어렵지만, 제품 담당자에게 문의하시면 가장 정확한 정보를 얻으실 수 있을 거예요. 혹시 담당자에게 연락할 방법이 있으시다면 그 길로 진행해 보시는 게 좋을 것 같습니다! 감사합니다.
